# Capstone — Refresh / Content Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Di-pesh/flyinterm/blob/main/work/notebooks/capstone.ipynb)

## Abstract

This project studies a practical editorial decision: which pages should a human review next for refresh or maintenance? We use the anonymized FlyRank starter dataset to rank pages by operational opportunity using freshness, visibility, and content metadata. We compare a transparent stale-and-visible rule with a logistic-regression model and evaluate both on the same client-grouped holdout. The notebook reports precision at several queue sizes, ROC AUC, and the test-set base rate without presenting retrospective association as causal evidence. The result is a reproducible review queue for human triage, not a claim that refresh work changes Google rankings or traffic.

## 1. Question and decision

**Research question:** which anonymized content pages should an editorial team review first for refresh or monitoring?

The unit is a content page. The output is a ranked queue with a score, action, and reason code. A human reviews the recommendation before making an editorial change. The cost of a false positive is wasted review time; the cost of a false negative is a potentially useful page being missed.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RAW_URL = 'https://raw.githubusercontent.com/Di-pesh/flyinterm/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(RAW_URL)
df = df.drop_duplicates('content_id').copy()
lane = df.loc[df['content_type'].eq('keyword article')].copy()
lane['target'] = lane['trend_direction'].astype('string').str.lower().eq('down').astype(int)
print(f'Raw rows: {len(df):,}')
print(f'Keyword-article lane rows: {len(lane):,}')
print(f'Lane target base rate: {lane["target"].mean():.3f}')


## 2. Data and safety

The source is `data/raw/content_refresh_anonymized.csv`, the public 30,000-row teaching snapshot described in `docs/data-dictionary.md`. It contains pseudonymized content items and aggregated trailing-window metrics. This capstone uses the keyword-article lane for a focused operational queue.

The retrospective target is `target = (trend_direction == 'down')`. Because `trend_direction` is computed from `trend_pct`, and `trend_pct` uses the recent and previous impression windows, neither is a feature. IDs are used only for grouping and tie-breaking. No client names, domains, URLs, titles, private queries, or credentials are used or printed.

In [ ]:
# Explicit leakage/privacy audit.
excluded = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct', 'target',
    'is_declining_label', 'impressions_last_30d', 'clicks_last_30d',
    'sessions_last_30d', 'impressions_90d', 'clicks_90d', 'pageviews_90d',
    'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
    'scroll_events_90d', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier'
}
print('Target construction verified:', lane['target'].equals(lane['trend_direction'].astype('string').str.lower().eq('down').astype(int)))
print('Excluded fields are reserved:', len(excluded))
print('Missingness by content type (word_count):')
display(lane.groupby('content_type', dropna=False)['word_count'].apply(lambda s: s.isna().mean()).rename('missing_rate').to_frame())


## 3. Methodology and baseline

The baseline says: **review pages that are both stale and visible**. Stale means `days_since_last_update >= 91`; visible means `impressions_90d >= 500`. The rule score is a readable combination of visibility and staleness. Each row receives a reason code and an action. This is a prioritization baseline, not a fitted model.

The model uses static content metadata and recency fields: search opportunity, content size, age, update recency, content type, intent, and age/freshness buckets. Missing numeric values receive a training-fold median plus a missingness flag; categorical missingness is represented as `unknown`. The target is retrospective and is used only for evaluation of ranking quality.

In [ ]:
# Transparent rule baseline.
lane['is_stale'] = lane['days_since_last_update'].fillna(0).ge(91)
lane['is_visible'] = lane['impressions_90d'].fillna(0).ge(500)
lane['baseline_score'] = np.log1p(lane['impressions_90d'].fillna(0)) * (1 + lane['is_stale'].astype(int))
lane['baseline_action'] = np.where(lane['is_stale'] & lane['is_visible'], 'refresh_review', 'monitor')
lane['baseline_reason'] = np.where(lane['is_stale'] & lane['is_visible'], 'stale_visible', 'not_both_stale_visible')
baseline_queue = lane.sort_values(['baseline_score', 'content_id'], ascending=[False, True]).reset_index(drop=True)
baseline_queue.insert(0, 'rank', np.arange(1, len(baseline_queue) + 1))
print('Baseline action counts:')
display(lane['baseline_action'].value_counts().rename_axis('action').to_frame('rows'))
display(baseline_queue[['rank', 'baseline_score', 'baseline_action', 'baseline_reason', 'days_since_last_update', 'impressions_90d']].head(10))


## 4. Grouped validation and model comparison

Rows from one client can share hidden characteristics. A random row split could let the model benefit from client-specific patterns, so the main evaluation holds out complete clients using `GroupShuffleSplit`. Baseline and model are scored on exactly the same test rows.

For a ranked queue, precision@K answers: among the first K recommendations, how many have the retrospective decline label? The base rate is printed alongside it. These are association and prioritization metrics, not evidence that a refresh would cause recovery.

In [ ]:
safe_numeric = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']
safe_categorical = ['content_type', 'main_intent', 'age_tier', 'freshness_tier']
X_raw = lane[safe_numeric + safe_categorical].copy()
for column in safe_numeric:
    X_raw[column] = pd.to_numeric(X_raw[column], errors='coerce')
    X_raw[f'has_{column}'] = X_raw[column].notna().astype(int)
for column in safe_categorical:
    X_raw[column] = X_raw[column].astype('string')
y = lane['target']
groups = lane['client_id'].astype('string').fillna('unknown')

numeric_columns = safe_numeric + [f'has_{c}' for c in safe_numeric]
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_columns),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), safe_categorical),
])
model = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))])

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X_raw, y, groups))
model.fit(X_raw.iloc[train_idx], y.iloc[train_idx])
model_scores = model.predict_proba(X_raw.iloc[test_idx])[:, 1]
baseline_scores = lane['baseline_score'].iloc[test_idx].to_numpy()
test_y = y.iloc[test_idx].to_numpy()

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))[:min(k, len(scores))]
    return float(np.asarray(labels)[order].mean())

metrics = []
for name, scores in [('Rule baseline', baseline_scores), ('Logistic model', model_scores)]:
    row = {'method': name, 'base_rate': float(test_y.mean()), 'roc_auc': float(roc_auc_score(test_y, scores)), 'average_precision': float(average_precision_score(test_y, scores))}
    for k in (10, 50, 100):
        row[f'precision_at_{k}'] = precision_at_k(test_y, scores, k)
    metrics.append(row)
metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(3))
print(f'Train rows: {len(train_idx):,}; test rows: {len(test_idx):,}; held-out clients: {groups.iloc[test_idx].nunique()}')


## 5. Leakage sentinel and interpretation

A valid harness should detect an intentionally leaky feature. The next cell evaluates `trend_pct`, which is part of the target construction and must never be used in the model. A very strong result is expected here; it is a test of the audit, not a result to report.

In [ ]:
leaky_score = pd.to_numeric(lane['trend_pct'], errors='coerce').fillna(0).to_numpy()[test_idx]
leak_auc = roc_auc_score(test_y, leaky_score)
print(f'Deliberate trend-derived leakage AUC: {leak_auc:.3f} (sentinel only; not a model result)')
assert leak_auc > 0.80, 'Leakage sentinel did not detect trend_pct; inspect the test harness.'
print('PASS: the deliberately label-derived field is detectable and excluded from X_raw.')


## 6. Limitations and honest framing

This is a retrospective starter-snapshot analysis, not a randomized content experiment. It cannot establish that refreshing a page causes more visibility, clicks, or engagement. The label is a trend association and the starter export does not provide a clean future intervention outcome. The client-grouped split reduces memorization risk but does not replace a time-aware production evaluation.

The safe claim is: the queue provides directional, observed decision support for human review in this dataset. It does not predict Google's algorithm, identify a causal ranking factor, or automate publishing decisions.

## 7. Ranked recommendations

1. Review the highest-ranked stale-and-visible pages first.
2. Check the page manually for seasonality, intentional evergreen status, and recent work not present in the snapshot.
3. Keep low-volume stale pages in a monitoring queue unless an editor has independent evidence of opportunity.
4. Use reason codes as a conversation starter, never as an automatic content instruction.
5. Re-run the queue after each editorial cycle and validate on a later time window before operational adoption.

## 8. Reproducibility and paper artifacts

The analysis can be rerun from the public repository with Python, pandas, NumPy, and scikit-learn. The random seed is 42. The core source is `data/raw/content_refresh_anonymized.csv`; the supporting notebooks are in `work/notebooks/`; and the final paper will be served from `docs/index.html`.

The paper must include the line: **Built on the [FlyRank ML Internship dataset](https://flyrank.ai)**.

In [ ]:
# Save non-sensitive metrics and a public-safe queue preview for the paper.
out_dir = 'work/outputs'
from pathlib import Path
Path(out_dir).mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(Path(out_dir) / 'capstone_metrics.csv', index=False)
baseline_queue[['rank', 'baseline_score', 'baseline_action', 'baseline_reason', 'days_since_last_update', 'impressions_90d']].head(100).to_csv(Path(out_dir) / 'capstone_queue_preview.csv', index=False)
print('Wrote work/outputs/capstone_metrics.csv')
print('Wrote work/outputs/capstone_queue_preview.csv')


## Self-check

- [x] Question, decision, and unit of analysis are explicit.
- [x] Data source and public-safety exclusions are documented.
- [x] Baseline is transparent and has reason codes.
- [x] Model uses a client-grouped holdout.
- [x] Baseline and model use the same test rows and metrics.
- [x] Base rate and precision-at-K are reported.
- [x] A deliberate leakage sentinel is included.
- [x] Limitations avoid causal and Google-algorithm claims.
- [ ] Run all cells top to bottom and save the executed notebook before submitting.